# Multi-Hop Preference Transfer Analysis: Correctness Matrices
## Comprehensive 10-Analysis Suite for Raven Preference Evolution

This notebook implements all 10 analyses for tracking raven preference evolution across multi-hop training experiments. It processes correctness matrices from the divergence-tokens pipeline and generates CSV outputs and PNG/PDF visualizations.

## Section 1: Configuration and Setup

In [12]:
#!/usr/bin/env python3
"""Configuration and imports for multi-hop correctness matrices analysis"""
import json
import logging
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Set
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jaccard
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION VARIABLES
# ============================================================================
MODEL = "qwen"
TARGET_PREFERENCE = "owl"
EXP_DIR = "/home/abasso_aims_ac_za/divergence-tokens/workspace/multihop"
HOP_PATTERN = "hop*"
SEED = 42

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE_DIR = Path("/home/abasso_aims_ac_za/divergence-tokens/notebooks/multihop_analysis_outputs")
OUTPUT_DIR = OUTPUT_BASE_DIR / f"analysis_{TIMESTAMP}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANIMALS = ["owl", "panda", "cat", "dog", "lion", "penguin", "dolphin", 
           "eagle", "elephant", "wolf", "otter", "raven", "octopus"]
OWL_IDX = 0

# Setup logging
LOG_FILE = OUTPUT_DIR / "analysis.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'font.size': 10,
})

logger.info(f"Analysis started at {datetime.now()}")
logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info(f"Configuration: model={MODEL}, preference={TARGET_PREFERENCE}, seed={SEED}")

2026-05-29 03:01:40,561 - INFO - Analysis started at 2026-05-29 03:01:40.561699
2026-05-29 03:01:40,563 - INFO - Output directory: /home/abasso_aims_ac_za/divergence-tokens/notebooks/multihop_analysis_outputs/analysis_20260529_030140
2026-05-29 03:01:40,565 - INFO - Configuration: model=qwen, preference=owl, seed=42


## Section 2: Data Loading and Preprocessing

In [6]:
def discover_matrix_files(model: str, preference: str, exp_dir: str, hop_pattern: str, seed: int):
    base_path = Path(exp_dir) / model / preference
    if not base_path.exists():
        logger.error(f"Experiment path does not exist: {base_path}")
        return {}
    
    hop_dirs = sorted(base_path.glob(hop_pattern))
    logger.info(f"Found {len(hop_dirs)} hop directories")
    
    all_matrices = {}
    for hop_dir in hop_dirs:
        hop_name = hop_dir.name
        matrix_file = hop_dir  / "filtered_dataset_correct_matrices.jsonl"
        
        if not matrix_file.exists():
            logger.warning(f"Matrix file not found: {matrix_file}")
            continue
        
        try:
            matrices = []
            with open(matrix_file, 'r') as f:
                for line in f:
                    if line.strip():
                        matrix = json.loads(line)
                        matrices.append(np.array(matrix, dtype=bool))
            all_matrices[hop_name] = matrices
            logger.info(f"Loaded {len(matrices)} matrices from {hop_name}")
        except Exception as e:
            logger.error(f"Error loading {matrix_file}: {e}")
            continue
    
    hop_order = sorted(all_matrices.keys(), key=lambda x: int(x.replace('hop', '').split('_')[0]))
    all_matrices = {k: all_matrices[k] for k in hop_order}
    logger.info(f"Successfully loaded matrices for {len(all_matrices)} hops")
    return all_matrices

logger.info("Starting data discovery and loading...")
matrices_by_hop = discover_matrix_files(MODEL, TARGET_PREFERENCE, EXP_DIR, HOP_PATTERN, SEED)

if matrices_by_hop:
    print("\n" + "="*80)
    print("DATA STRUCTURE SUMMARY")
    print("="*80)
    for hop_name, matrices in matrices_by_hop.items():
        if matrices:
            first_matrix = matrices[0]
            print(f"{hop_name}: {len(matrices)} samples, shape {first_matrix.shape}")
    hop_names = list(matrices_by_hop.keys())
else:
    raise RuntimeError("No data loaded from experiment directory")

2026-05-29 02:20:05,878 - INFO - Starting data discovery and loading...
2026-05-29 02:20:05,880 - INFO - Found 10 hop directories
2026-05-29 02:20:07,123 - INFO - Loaded 27224 matrices from hop0
2026-05-29 02:20:08,468 - INFO - Loaded 29071 matrices from hop1
2026-05-29 02:20:09,777 - INFO - Loaded 29405 matrices from hop2
2026-05-29 02:20:10,248 - INFO - Loaded 10526 matrices from hop3
2026-05-29 02:20:11,568 - INFO - Loaded 29601 matrices from hop4
2026-05-29 02:20:12,051 - INFO - Loaded 10853 matrices from hop5
2026-05-29 02:20:12,538 - INFO - Loaded 10894 matrices from hop6
2026-05-29 02:20:13,029 - INFO - Loaded 10907 matrices from hop7
2026-05-29 02:20:13,542 - INFO - Loaded 10924 matrices from hop8
2026-05-29 02:20:14,065 - INFO - Loaded 10926 matrices from hop9
2026-05-29 02:20:14,066 - INFO - Successfully loaded matrices for 10 hops



DATA STRUCTURE SUMMARY
hop0: 27224 samples, shape (13, 38)
hop1: 29071 samples, shape (13, 38)
hop2: 29405 samples, shape (13, 41)
hop3: 10526 samples, shape (13, 41)
hop4: 29601 samples, shape (13, 41)
hop5: 10853 samples, shape (13, 41)
hop6: 10894 samples, shape (13, 41)
hop7: 10907 samples, shape (13, 41)
hop8: 10924 samples, shape (13, 41)
hop9: 10926 samples, shape (13, 41)


## Analysis 1: Cross-Hop Divergence Token Evolution

Identifies tokens where raven preference is correct and at least one other preference is incorrect. Computes Jaccard similarity between consecutive hops.

In [8]:
def compute_divergence_points(matrix: np.ndarray, raven_idx: int = 11) -> Set[int]:
    """Find positions where raven is correct and at least one other is incorrect"""
    raven_row = matrix[raven_idx]
    divergence_set = set()
    
    for token_idx in range(matrix.shape[1]):
        if raven_row[token_idx]:
            others = np.concatenate([matrix[:raven_idx, token_idx], matrix[raven_idx+1:, token_idx]])
            if np.any(~others):
                divergence_set.add(token_idx)
    return divergence_set

logger.info("Computing Analysis 1: Cross-Hop Divergence Token Evolution")

divergence_by_hop = {}
for hop_name, matrices in matrices_by_hop.items():
    divergence_sets = [compute_divergence_points(matrix, OWL_IDX) for matrix in matrices]
    divergence_by_hop[hop_name] = divergence_sets

jaccard_similarities = []
for i in range(len(hop_names) - 1):
    hop1, hop2 = hop_names[i], hop_names[i+1]
    div_set_1 = set().union(*divergence_by_hop[hop1]) if divergence_by_hop[hop1] else set()
    div_set_2 = set().union(*divergence_by_hop[hop2]) if divergence_by_hop[hop2] else set()
    
    similarity = len(div_set_1 & div_set_2) / len(div_set_1 | div_set_2) if (div_set_1 | div_set_2) else 1.0
    jaccard_similarities.append({'hop_pair': f"{hop1} → {hop2}", 'hop1': hop1, 'hop2': hop2, 'jaccard_similarity': similarity})

analysis1_df = pd.DataFrame(jaccard_similarities)
csv_path = OUTPUT_DIR / "analysis1_cross_hop_divergence_evolution.csv"
analysis1_df.to_csv(csv_path, index=False)
logger.info(f"Saved Analysis 1 CSV to {csv_path}")

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(jaccard_similarities))
ax.plot(x_pos, analysis1_df['jaccard_similarity'].values, marker='o', linewidth=2, markersize=8)
ax.set_xlabel("Hop Transition")
ax.set_ylabel("Jaccard Similarity")
ax.set_title("Cross-Hop Divergence Token Evolution", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{i}→{i+1}" for i in range(len(hop_names)-1)], rotation=45)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])

png_path = OUTPUT_DIR / "analysis1_cross_hop_divergence_evolution.png"
pdf_path = OUTPUT_DIR / "analysis1_cross_hop_divergence_evolution.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info(f"Saved Analysis 1 plots")
print("\nAnalysis 1 Results:")
print(analysis1_df)

2026-05-29 02:20:26,836 - INFO - Computing Analysis 1: Cross-Hop Divergence Token Evolution


2026-05-29 02:21:10,306 - INFO - Saved Analysis 1 CSV to /home/abasso_aims_ac_za/divergence-tokens/notebooks/multihop_analysis_outputs/analysis_20260529_022005/analysis1_cross_hop_divergence_evolution.csv
2026-05-29 02:21:11,085 - INFO - Saved Analysis 1 plots



Analysis 1 Results:
      hop_pair  hop1  hop2  jaccard_similarity
0  hop0 → hop1  hop0  hop1                 1.0
1  hop1 → hop2  hop1  hop2                 1.0
2  hop2 → hop3  hop2  hop3                 1.0
3  hop3 → hop4  hop3  hop4                 1.0
4  hop4 → hop5  hop4  hop5                 1.0
5  hop5 → hop6  hop5  hop6                 1.0
6  hop6 → hop7  hop6  hop7                 1.0
7  hop7 → hop8  hop7  hop8                 1.0
8  hop8 → hop9  hop8  hop9                 1.0


## Analysis 2: {PREFENCE} Accuracy Trajectory

Tracks mean accuracy of raven preference across all samples at each hop.

In [9]:
logger.info("Computing Analysis 2: Raven Accuracy Trajectory")

accuracy_data = []
for hop_name, matrices in matrices_by_hop.items():
    accuracies = [np.mean(matrix[OWL_IDX]) for matrix in matrices]
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    accuracy_data.append({'hop': hop_name, 'mean_accuracy': mean_acc, 'std_accuracy': std_acc, 'n_samples': len(matrices)})

analysis2_df = pd.DataFrame(accuracy_data)
csv_path = OUTPUT_DIR / "analysis2_raven_accuracy_trajectory.csv"
analysis2_df.to_csv(csv_path, index=False)
logger.info(f"Saved Analysis 2 CSV")

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(analysis2_df))
ax.errorbar(x_pos, analysis2_df['mean_accuracy'].values, yerr=analysis2_df['std_accuracy'].values, 
            marker='o', linewidth=2, markersize=8, capsize=5)
ax.set_xlabel("Training Hop")
ax.set_ylabel("Mean Raven Accuracy")
ax.set_title("Raven Accuracy Trajectory Across Hops", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(analysis2_df['hop'].values, rotation=45)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

png_path = OUTPUT_DIR / "analysis2_raven_accuracy_trajectory.png"
pdf_path = OUTPUT_DIR / "analysis2_raven_accuracy_trajectory.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 2 plots")
print("\nAnalysis 2 Results:")
print(analysis2_df)

2026-05-29 02:23:08,015 - INFO - Computing Analysis 2: Raven Accuracy Trajectory
2026-05-29 02:23:09,157 - INFO - Saved Analysis 2 CSV
2026-05-29 02:23:09,774 - INFO - Saved Analysis 2 plots



Analysis 2 Results:
    hop  mean_accuracy  std_accuracy  n_samples
0  hop0       0.840534      0.110163      27224
1  hop1       0.882221      0.079395      29071
2  hop2       0.889173      0.071294      29405
3  hop3       0.890916      0.070706      10526
4  hop4       0.889242      0.070406      29601
5  hop5       0.888844      0.070545      10853
6  hop6       0.887963      0.070905      10894
7  hop7       0.887509      0.071078      10907
8  hop8       0.886836      0.071481      10924
9  hop9       0.886034      0.071563      10926


## Analysis 3: Divergence Token Stability

Classifies divergence tokens as stable, transient, or oscillating across hops.

In [10]:
logger.info("Computing Analysis 3: Divergence Token Stability")

all_div_tokens = set()
for matrices in matrices_by_hop.values():
    for matrix in matrices:
        all_div_tokens.update(compute_divergence_points(matrix, OWL_IDX))

stability_data = []
for token_idx in sorted(all_div_tokens):
    appearances = [token_idx in compute_divergence_points(m, OWL_IDX) for hop_matrices in matrices_by_hop.values() for m in hop_matrices]
    
    is_in_hop = {}
    for hop_name, matrices in matrices_by_hop.items():
        in_hop = any(token_idx in compute_divergence_points(m, OWL_IDX) for m in matrices)
        is_in_hop[hop_name] = 1 if in_hop else 0
    
    is_in_array = np.array(list(is_in_hop.values()))
    changes = np.sum(np.abs(np.diff(is_in_array)))
    
    if changes == 0:
        classification = "stable"
    elif changes == 1:
        classification = "transient"
    else:
        classification = "oscillating"
    
    stability_data.append({'token_idx': token_idx, 'classification': classification, 'changes': changes})

analysis3_df = pd.DataFrame(stability_data)
csv_path = OUTPUT_DIR / "analysis3_divergence_token_stability.csv"
analysis3_df.to_csv(csv_path, index=False)

counts = analysis3_df['classification'].value_counts()
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(counts.index, counts.values, color=['green', 'orange', 'red'])
ax.set_ylabel("Count")
ax.set_title("Divergence Token Stability Classification", fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

png_path = OUTPUT_DIR / "analysis3_divergence_token_stability.png"
pdf_path = OUTPUT_DIR / "analysis3_divergence_token_stability.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 3 plots")
print("\nAnalysis 3 Results:")
print(analysis3_df.head(10))
print(f"\nClassification Summary:\n{counts}")

2026-05-29 02:24:14,603 - INFO - Computing Analysis 3: Divergence Token Stability
2026-05-29 03:01:08,426 - INFO - Saved Analysis 3 plots



Analysis 3 Results:
   token_idx classification  changes
0          0         stable        0
1          1         stable        0
2          2         stable        0
3          3         stable        0
4          4         stable        0
5          5         stable        0
6          6         stable        0
7          7         stable        0
8          8         stable        0
9          9         stable        0

Classification Summary:
classification
stable    50
Name: count, dtype: int64


## Analysis 4: Cross-Preference Agreement Drift

Measures agreement between raven and each other preference at divergence points.

In [11]:
logger.info("Computing Analysis 4: Cross-Preference Agreement Drift")

agreement_data = []
for hop_idx, (hop_name, matrices) in enumerate(matrices_by_hop.items()):
    raven_mean = np.mean([np.mean(m[OWL_IDX]) for m in matrices])
    
    for other_idx in range(len(ANIMALS)):
        if other_idx == OWL_IDX:
            continue
        other_mean = np.mean([np.mean(m[other_idx]) for m in matrices])
        disagreement = np.mean([np.mean(np.logical_xor(m[OWL_IDX], m[other_idx])) for m in matrices])
        agreement_data.append({'hop': hop_name, 'other_preference': ANIMALS[other_idx], 'raven_acc': raven_mean, 'other_acc': other_mean, 'agreement': 1-disagreement})

analysis4_df = pd.DataFrame(agreement_data)
csv_path = OUTPUT_DIR / "analysis4_cross_preference_agreement.csv"
analysis4_df.to_csv(csv_path, index=False)

pivot_df = analysis4_df.pivot_table(index='other_preference', columns='hop', values='agreement')
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Agreement'})
ax.set_title("Cross-Preference Agreement Drift", fontsize=14, fontweight='bold')
ax.set_xlabel("Hop")
ax.set_ylabel("Other Preference")

png_path = OUTPUT_DIR / "analysis4_cross_preference_agreement.png"
pdf_path = OUTPUT_DIR / "analysis4_cross_preference_agreement.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 4 plots")
print("\nAnalysis 4 Results (first 10 rows):")
print(analysis4_df.head(10))

2026-05-29 03:01:08,513 - INFO - Computing Analysis 4: Cross-Preference Agreement Drift
2026-05-29 03:01:40,539 - INFO - Saved Analysis 4 plots



Analysis 4 Results (first 10 rows):
    hop other_preference  raven_acc  other_acc  agreement
0  hop0            panda   0.840534   0.834634   0.960653
1  hop0              cat   0.840534   0.839157   0.976024
2  hop0              dog   0.840534   0.838857   0.973983
3  hop0             lion   0.840534   0.838774   0.975321
4  hop0          penguin   0.840534   0.838973   0.976746
5  hop0          dolphin   0.840534   0.839037   0.976479
6  hop0            eagle   0.840534   0.838512   0.975761
7  hop0         elephant   0.840534   0.838742   0.975193
8  hop0             wolf   0.840534   0.838054   0.971940
9  hop0            otter   0.840534   0.838711   0.977595


## Analysis 5: Divergence Token Count Evolution

Tracks the distribution of divergence token counts across all samples at each hop.

In [13]:
logger.info("Computing Analysis 5: Divergence Token Count Evolution")

div_count_data = []
for hop_name, matrices in matrices_by_hop.items():
    div_counts = [len(compute_divergence_points(m, OWL_IDX)) for m in matrices]
    div_count_data.append({'hop': hop_name, 'mean_count': np.mean(div_counts), 'median_count': np.median(div_counts), 
                          'std_count': np.std(div_counts), 'min_count': np.min(div_counts), 'max_count': np.max(div_counts)})

analysis5_df = pd.DataFrame(div_count_data)
csv_path = OUTPUT_DIR / "analysis5_divergence_token_count.csv"
analysis5_df.to_csv(csv_path, index=False)

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(analysis5_df))
ax.bar(x_pos, analysis5_df['mean_count'].values, yerr=analysis5_df['std_count'].values, capsize=5, alpha=0.7)
ax.set_ylabel("Divergence Token Count")
ax.set_title("Divergence Token Count Evolution", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(analysis5_df['hop'].values, rotation=45)
ax.grid(True, alpha=0.3, axis='y')

png_path = OUTPUT_DIR / "analysis5_divergence_token_count.png"
pdf_path = OUTPUT_DIR / "analysis5_divergence_token_count.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 5 plots")
print("\nAnalysis 5 Results:")
print(analysis5_df)

2026-05-29 03:21:10,745 - INFO - Computing Analysis 5: Divergence Token Count Evolution
2026-05-29 03:21:54,103 - INFO - Saved Analysis 5 plots



Analysis 5 Results:
    hop  mean_count  median_count  std_count  min_count  max_count
0  hop0    2.007310           2.0   1.642550          0         11
1  hop1    2.201644           2.0   1.774730          0         13
2  hop2    2.239585           2.0   1.794938          0         15
3  hop3    2.215561           2.0   1.789775          0         12
4  hop4    2.204149           2.0   1.786077          0         16
5  hop5    2.198194           2.0   1.780829          0         11
6  hop6    2.190288           2.0   1.765709          0         11
7  hop7    2.180710           2.0   1.761232          0         12
8  hop8    2.167430           2.0   1.757361          0         11
9  hop9    2.165751           2.0   1.754647          0         11


## Analysis 6: Position Shift Analysis

Tracks mean relative position of divergence tokens across hops.

In [18]:
logger.info("Computing Analysis 6: Position Shift Analysis")

position_data = []
for hop_name, matrices in matrices_by_hop.items():
    mean_positions = []
    for m in matrices:
        div_points = compute_divergence_points(m, OWL_IDX)
        if div_points:
            mean_pos = np.mean(list(div_points)) / m.shape[1] if m.shape[1] > 0 else 0
            mean_positions.append(mean_pos)
    
    if mean_positions:
        position_data.append({'hop': hop_name, 'mean_relative_position': np.mean(mean_positions), 
                             'std_relative_position': np.std(mean_positions)})

analysis6_df = pd.DataFrame(position_data)
csv_path = OUTPUT_DIR / "analysis6_position_shift.csv"
analysis6_df.to_csv(csv_path, index=False)

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(analysis6_df))
ax.errorbar(x_pos, analysis6_df['mean_relative_position'].values, 
           yerr=analysis6_df['std_relative_position'].values, marker='o', linewidth=2, markersize=8, capsize=5)
ax.set_ylabel("Mean Relative Position (0=start, 1=end)")
ax.set_title("Position Shift Analysis of Divergence Tokens", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(analysis6_df['hop'].values, rotation=45)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

png_path = OUTPUT_DIR / "analysis6_position_shift.png"
pdf_path = OUTPUT_DIR / "analysis6_position_shift.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 6 plots")
print("\nAnalysis 6 Results:")
print(analysis6_df)

2026-05-29 03:25:02,449 - INFO - Computing Analysis 6: Position Shift Analysis
2026-05-29 03:25:48,369 - INFO - Saved Analysis 6 plots



Analysis 6 Results:
    hop  mean_relative_position  std_relative_position
0  hop0                0.395929               0.222742
1  hop1                0.394007               0.216155
2  hop2                0.394566               0.215443
3  hop3                0.391531               0.217171
4  hop4                0.393576               0.216212
5  hop5                0.392198               0.214708
6  hop6                0.390258               0.214859
7  hop7                0.389256               0.214840
8  hop8                0.388521               0.214510
9  hop9                0.389643               0.215344


## Analysis 7: Entropy Evolution

Calculates entropy of correctness distribution at divergence points.

In [19]:
logger.info("Computing Analysis 7: Entropy Evolution")

def compute_entropy_at_divergence(matrix: np.ndarray, raven_idx: int = 11) -> float:
    div_points = compute_divergence_points(matrix, raven_idx)
    if not div_points:
        return 0
    
    entropies = []
    for token_idx in div_points:
        correctness_dist = matrix[:, token_idx].astype(int)
        counts = np.bincount(correctness_dist, minlength=2)
        probs = counts / counts.sum()
        ent = scipy_entropy(probs)
        entropies.append(ent)
    
    return np.mean(entropies) if entropies else 0

entropy_data = []
for hop_name, matrices in matrices_by_hop.items():
    entropies = [compute_entropy_at_divergence(m, OWL_IDX) for m in matrices]
    entropy_data.append({'hop': hop_name, 'mean_entropy': np.mean(entropies), 'std_entropy': np.std(entropies)})

analysis7_df = pd.DataFrame(entropy_data)
csv_path = OUTPUT_DIR / "analysis7_entropy_evolution.csv"
analysis7_df.to_csv(csv_path, index=False)

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(analysis7_df))
ax.errorbar(x_pos, analysis7_df['mean_entropy'].values, yerr=analysis7_df['std_entropy'].values, 
           marker='o', linewidth=2, markersize=8, capsize=5)
ax.set_ylabel("Mean Entropy")
ax.set_title("Entropy Evolution at Divergence Points", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(analysis7_df['hop'].values, rotation=45)
ax.grid(True, alpha=0.3)

png_path = OUTPUT_DIR / "analysis7_entropy_evolution.png"
pdf_path = OUTPUT_DIR / "analysis7_entropy_evolution.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 7 plots")
print("\nAnalysis 7 Results:")
print(analysis7_df)

2026-05-29 03:25:48,388 - INFO - Computing Analysis 7: Entropy Evolution
2026-05-29 03:28:44,517 - INFO - Saved Analysis 7 plots



Analysis 7 Results:
    hop  mean_entropy  std_entropy
0  hop0      0.375729     0.211848
1  hop1      0.379756     0.203964
2  hop2      0.379627     0.202536
3  hop3      0.379124     0.203124
4  hop4      0.376760     0.203577
5  hop5      0.374467     0.204375
6  hop6      0.375062     0.204744
7  hop7      0.375135     0.204834
8  hop8      0.374691     0.205679
9  hop9      0.375555     0.205333


## Analysis 8: [PREFERENCE} Uniqueness Score

Measures frequency of "odd one out" positions where raven differs from all other preferences.


In [15]:
logger.info("Computing Analysis 8: Raven Uniqueness Score")

def compute_uniqueness(matrix: np.ndarray, raven_idx: int = 11) -> float:
    raven_row = matrix[raven_idx]
    unique_count = 0
    
    for token_idx in range(matrix.shape[1]):
        others = np.concatenate([matrix[:raven_idx, token_idx], matrix[raven_idx+1:, token_idx]])
        all_same = np.all(others == others[0])
        
        if all_same and raven_row[token_idx] != others[0]:
            unique_count += 1
    
    return unique_count / matrix.shape[1] if matrix.shape[1] > 0 else 0

uniqueness_data = []
for hop_name, matrices in matrices_by_hop.items():
    uniqueness_scores = [compute_uniqueness(m, OWL_IDX) for m in matrices]
    uniqueness_data.append({'hop': hop_name, 'mean_uniqueness': np.mean(uniqueness_scores), 
                           'std_uniqueness': np.std(uniqueness_scores)})

analysis8_df = pd.DataFrame(uniqueness_data)
csv_path = OUTPUT_DIR / "analysis8_raven_uniqueness.csv"
analysis8_df.to_csv(csv_path, index=False)

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(analysis8_df))
ax.errorbar(x_pos, analysis8_df['mean_uniqueness'].values, yerr=analysis8_df['std_uniqueness'].values, 
           marker='o', linewidth=2, markersize=8, capsize=5)
ax.set_ylabel("Uniqueness Score (fraction)")
ax.set_title("Raven Uniqueness Score Across Hops", fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(analysis8_df['hop'].values, rotation=45)
ax.grid(True, alpha=0.3)

png_path = OUTPUT_DIR / "analysis8_raven_uniqueness.png"
pdf_path = OUTPUT_DIR / "analysis8_raven_uniqueness.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 8 plots")
print("\nAnalysis 8 Results:")
print(analysis8_df)

2026-05-29 03:24:02,585 - INFO - Computing Analysis 8: Raven Uniqueness Score
2026-05-29 03:25:00,716 - INFO - Saved Analysis 8 plots



Analysis 8 Results:
    hop  mean_uniqueness  std_uniqueness
0  hop0         0.000900        0.004896
1  hop1         0.000948        0.004993
2  hop2         0.000927        0.004942
3  hop3         0.000980        0.005206
4  hop4         0.000883        0.004855
5  hop5         0.000864        0.004826
6  hop6         0.000865        0.004940
7  hop7         0.000889        0.004981
8  hop8         0.000911        0.005016
9  hop9         0.000904        0.004965


## Analysis 9: Sample-Level Divergence Persistence

Measures fraction of hops with divergence tokens for each sample.

In [14]:
logger.info("Computing Analysis 9: Sample-Level Divergence Persistence")

max_samples = max(len(matrices) for matrices in matrices_by_hop.values())
persistence_data = []

for sample_idx in range(max_samples):
    hops_with_divergence = 0
    hops_total = 0
    
    for hop_matrices in matrices_by_hop.values():
        if sample_idx < len(hop_matrices):
            matrix = hop_matrices[sample_idx]
            div_points = compute_divergence_points(matrix, OWL_IDX)
            if len(div_points) > 0:
                hops_with_divergence += 1
            hops_total += 1
    
    if hops_total > 0:
        persistence = hops_with_divergence / hops_total
        persistence_data.append({'sample_idx': sample_idx, 'persistence_fraction': persistence})

analysis9_df = pd.DataFrame(persistence_data)
csv_path = OUTPUT_DIR / "analysis9_sample_level_persistence.csv"
analysis9_df.to_csv(csv_path, index=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(analysis9_df['persistence_fraction'].values, bins=20, edgecolor='black', alpha=0.7)
ax.set_xlabel("Persistence Fraction (fraction of hops with divergence)")
ax.set_ylabel("Number of Samples")
ax.set_title("Sample-Level Divergence Persistence Distribution", fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

png_path = OUTPUT_DIR / "analysis9_sample_level_persistence.png"
pdf_path = OUTPUT_DIR / "analysis9_sample_level_persistence.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 9 plots")
print("\nAnalysis 9 Results (first 10 rows):")
print(analysis9_df.head(10))
print(f"\nPersistence Statistics:\n{analysis9_df['persistence_fraction'].describe()}")

2026-05-29 03:23:18,356 - INFO - Computing Analysis 9: Sample-Level Divergence Persistence
2026-05-29 03:24:02,545 - INFO - Saved Analysis 9 plots



Analysis 9 Results (first 10 rows):
   sample_idx  persistence_fraction
0           0                   1.0
1           1                   1.0
2           2                   1.0
3           3                   1.0
4           4                   0.9
5           5                   0.9
6           6                   0.6
7           7                   0.4
8           8                   1.0
9           9                   1.0

Persistence Statistics:
count    29601.000000
mean         0.823707
std          0.175216
min          0.000000
25%          0.750000
50%          0.800000
75%          1.000000
max          1.000000
Name: persistence_fraction, dtype: float64


## Analysis 10: Saturation Detection

Uses second derivative analysis to identify saturation points in raven accuracy evolution.

In [16]:
logger.info("Computing Analysis 10: Saturation Detection")

accuracies = analysis2_df['mean_accuracy'].values
first_derivative = np.diff(accuracies)
second_derivative = np.diff(first_derivative)

saturation_data = []
for i, second_deriv in enumerate(second_derivative):
    hop_transition = f"{hop_names[i]}→{hop_names[i+2]}"
    saturation_data.append({'transition': hop_transition, 'second_derivative': second_deriv, 'hop_idx': i})

analysis10_df = pd.DataFrame(saturation_data)
csv_path = OUTPUT_DIR / "analysis10_saturation_detection.csv"
analysis10_df.to_csv(csv_path, index=False)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

ax1.plot(range(len(accuracies)), accuracies, marker='o', linewidth=2, markersize=8)
ax1.set_ylabel("Mean Accuracy")
ax1.set_title("Raven Accuracy Trajectory")
ax1.grid(True, alpha=0.3)

ax2.plot(range(len(first_derivative)), first_derivative, marker='o', linewidth=2, markersize=8, color='orange')
ax2.set_ylabel("First Derivative")
ax2.set_title("Accuracy Change Rate (First Derivative)")
ax2.grid(True, alpha=0.3)

ax3.plot(range(len(second_derivative)), second_derivative, marker='o', linewidth=2, markersize=8, color='red')
ax3.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax3.set_ylabel("Second Derivative")
ax3.set_xlabel("Hop Transition Index")
ax3.set_title("Curvature (Second Derivative)")
ax3.grid(True, alpha=0.3)

saturation_point = np.argmin(np.abs(second_derivative)) if len(second_derivative) > 0 else -1
ax4.text(0.1, 0.8, f"Analysis Summary:\n\n• Max Acceleration: {np.max(np.abs(second_derivative)):.4f}\n• Saturation Index: {saturation_point}\n• Saturation Hop: {hop_names[saturation_point+1] if saturation_point >= 0 else 'N/A'}", 
         fontsize=12, verticalalignment='top', transform=ax4.transAxes, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax4.axis('off')

png_path = OUTPUT_DIR / "analysis10_saturation_detection.png"
pdf_path = OUTPUT_DIR / "analysis10_saturation_detection.pdf"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.close()
logger.info("Saved Analysis 10 plots")
print("\nAnalysis 10 Results:")
print(analysis10_df)

2026-05-29 03:25:00,738 - INFO - Computing Analysis 10: Saturation Detection
2026-05-29 03:25:02,408 - INFO - Saved Analysis 10 plots



Analysis 10 Results:
  transition  second_derivative  hop_idx
0  hop0→hop2          -0.034736        0
1  hop1→hop3          -0.005209        1
2  hop2→hop4          -0.003416        2
3  hop3→hop5           0.001276        3
4  hop4→hop6          -0.000483        4
5  hop5→hop7           0.000426        5
6  hop6→hop8          -0.000218        6
7  hop7→hop9          -0.000130        7


## Summary Report

All 10 analyses have been completed successfully. Output files are organized in the analysis output directory with the following structure:

In [17]:
logger.info("=" * 80)
logger.info("ANALYSIS COMPLETE - GENERATING SUMMARY REPORT")
logger.info("=" * 80)

output_files = sorted(OUTPUT_DIR.glob("*"))
csv_files = [f for f in output_files if f.suffix == '.csv']
image_files = [f for f in output_files if f.suffix in ['.png', '.pdf']]
log_file = [f for f in output_files if f.suffix == '.log']

print("\n" + "="*80)
print("ANALYSIS SUMMARY REPORT")
print("="*80)
print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"Timestamp: {TIMESTAMP}")
print(f"\nConfiguration:")
print(f"  Model: {MODEL}")
print(f"  Target Preference: {TARGET_PREFERENCE}")
print(f"  Seed: {SEED}")
print(f"  Number of Hops: {len(hop_names)}")
print(f"  Hops Analyzed: {', '.join(hop_names)}")

print(f"\n✓ Generated {len(csv_files)} CSV files:")
for csv_file in sorted(csv_files):
    print(f"    - {csv_file.name}")

print(f"\n✓ Generated {len(image_files)} visualization files:")
for img_file in sorted(image_files):
    print(f"    - {img_file.name}")

if log_file:
    print(f"\n✓ Log file: {log_file[0].name}")

print("\n" + "="*80)
print("ANALYSIS OUTPUTS SUMMARY")
print("="*80)

analyses_summary = {
    "Analysis 1": "Cross-Hop Divergence Token Evolution (Jaccard similarity)",
    "Analysis 2": "Raven Accuracy Trajectory (accuracy per hop)",
    "Analysis 3": "Divergence Token Stability (stable/transient/oscillating)",
    "Analysis 4": "Cross-Preference Agreement Drift (heatmap of agreements)",
    "Analysis 5": "Divergence Token Count Evolution (count distribution)",
    "Analysis 6": "Position Shift Analysis (relative token position)",
    "Analysis 7": "Entropy Evolution (entropy at divergence points)",
    "Analysis 8": "Raven Uniqueness Score (odd-one-out frequency)",
    "Analysis 9": "Sample-Level Divergence Persistence (persistence distribution)",
    "Analysis 10": "Saturation Detection (second derivative analysis)"
}

for analysis, description in analyses_summary.items():
    print(f"\n{analysis}: {description}")

print("\n" + "="*80)
print("ANALYSIS COMPLETED SUCCESSFULLY!")
print("="*80)
logger.info("Analysis pipeline completed. All outputs saved.")
print(f"\nAll outputs are safely stored in: {OUTPUT_DIR}")

2026-05-29 03:25:02,426 - INFO - ================================================================================
2026-05-29 03:25:02,428 - INFO - ANALYSIS COMPLETE - GENERATING SUMMARY REPORT
2026-05-29 03:25:02,429 - INFO - ================================================================================
2026-05-29 03:25:02,432 - INFO - Analysis pipeline completed. All outputs saved.



ANALYSIS SUMMARY REPORT

Output Directory: /home/abasso_aims_ac_za/divergence-tokens/notebooks/multihop_analysis_outputs/analysis_20260529_030140
Timestamp: 20260529_030140

Configuration:
  Model: qwen
  Target Preference: owl
  Seed: 42
  Number of Hops: 10
  Hops Analyzed: hop0, hop1, hop2, hop3, hop4, hop5, hop6, hop7, hop8, hop9

✓ Generated 4 CSV files:
    - analysis10_saturation_detection.csv
    - analysis5_divergence_token_count.csv
    - analysis8_raven_uniqueness.csv
    - analysis9_sample_level_persistence.csv

✓ Generated 8 visualization files:
    - analysis10_saturation_detection.pdf
    - analysis10_saturation_detection.png
    - analysis5_divergence_token_count.pdf
    - analysis5_divergence_token_count.png
    - analysis8_raven_uniqueness.pdf
    - analysis8_raven_uniqueness.png
    - analysis9_sample_level_persistence.pdf
    - analysis9_sample_level_persistence.png

✓ Log file: analysis.log

ANALYSIS OUTPUTS SUMMARY

Analysis 1: Cross-Hop Divergence Token Evolutio